# 🌌 Aurora Activity — Linear Regression Prediction & Forecasting
**Models:** Linear Regression · Ridge · Lasso · ElasticNet · Polynomial Ridge · Huber

**Targets:** Kp Index · Ap Index · Aurora Probability

**Features:** Kp1-Kp8 (3-hourly), ap1-ap8, Ap, SN (sunspot number), F10.7 solar flux + engineered features

---
### Fixes & Improvements in this version
- ✅ Fixed `NameError: 'df' is not defined` — data loading now runs inline and all cells are self-contained
- ✅ Output directory auto-created relative to dataset (`v1-geo outputs/` subfolder)
- ✅ Graceful fallback: downloads GFZ Kp data automatically if no local files found
- ✅ Improved forecast accuracy: GridSearchCV hyperparameter tuning, better ensemble weighting
- ✅ Added `BayesianRidge` model — provides native prediction intervals
- ✅ NOAA/NASA comparison panel — fetches live 3-day Kp forecast from NOAA SWPC API with fallback
- ✅ Calibrated aurora probability using isotonic regression
- ✅ Richer feature engineering: 27-day solar rotation lags, Dst proxy, storm recovery features
- ✅ All plots saved to output subfolder automatically

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
import subprocess, sys
pkgs = ['scikit-learn', 'matplotlib', 'pandas', 'numpy', 'seaborn', 'scipy', 'joblib', 'requests']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('All dependencies installed ✓')

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import os
import glob
import warnings
import urllib.request
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path

try:
    import requests
    REQUESTS_OK = True
except ImportError:
    REQUESTS_OK = False

from sklearn.preprocessing import MinMaxScaler, RobustScaler, PolynomialFeatures
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    BayesianRidge, HuberRegressor
)
from sklearn.isotonic import IsotonicRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    precision_score, recall_score, f1_score, brier_score_loss
)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.inspection import permutation_importance
import joblib
import scipy.stats as stats

warnings.filterwarnings('ignore')
np.random.seed(42)

print('All libraries loaded successfully ✓')

## Configuration

In [ ]:

DATA_DIR = None  

LOOKBACK        = 30        
FORECAST_DAYS   = 30        
TEST_SPLIT      = 0.15     
VAL_SPLIT       = 0.15      

AURORA_KP_THRESHOLD = 5.0   
TARGET_COL = 'Kp_mean'      

COLORS = {
    'Linear Regression': '#3b82f6',
    'Ridge':             '#10b981',
    'Lasso':             '#f59e0b',
    'ElasticNet':        '#8b5cf6',
    'Polynomial Ridge':  '#ef4444',
    'Huber':             '#06b6d4',
    'BayesianRidge':     '#ec4899',
    'Actual':            '#1e293b',
}

print('Configuration set ✓')

## Data Loading & Parsing

In [ ]:
def parse_kp_file(filepath_or_text):

    if isinstance(filepath_or_text, (str, Path)) and Path(filepath_or_text).exists():
        with open(filepath_or_text, 'r', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
    else:
        # treat as raw text
        lines = str(filepath_or_text).splitlines()

    rows = []
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) < 26:
            continue
        try:
            year, month, day = int(parts[0]), int(parts[1]), int(parts[2])
            if not (1932 <= year <= datetime.now().year):
                continue
            kp      = [float(parts[6+i])  for i in range(8)]
            ap      = [int(parts[14+i])   for i in range(8)]
            Ap      = int(parts[22])
            SN      = int(parts[23])
            f107obs = float(parts[24])
            f107adj = float(parts[25])

            rows.append({
                'date':    datetime(year, month, day),
                'Kp_mean': sum(kp) / 8.0,
                'Kp1': kp[0], 'Kp2': kp[1], 'Kp3': kp[2], 'Kp4': kp[3],
                'Kp5': kp[4], 'Kp6': kp[5], 'Kp7': kp[6], 'Kp8': kp[7],
                'ap1': ap[0], 'ap2': ap[1], 'ap3': ap[2], 'ap4': ap[3],
                'ap5': ap[4], 'ap6': ap[5], 'ap7': ap[6], 'ap8': ap[7],
                'Ap': Ap, 'SN': SN,
                'F107obs': f107obs, 'F107adj': f107adj
            })
        except (ValueError, IndexError):
            continue
    return pd.DataFrame(rows)


def download_gfz_data():
   
    url = 'https://kp.gfz-potsdam.de/app/files/Kp_ap_Ap_SN_F107_since_1932.txt'
    print(f'Downloading GFZ Kp data from {url} ...')
    try:
        if REQUESTS_OK:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            text = resp.text
        else:
            with urllib.request.urlopen(url, timeout=60) as r:
                text = r.read().decode('utf-8', errors='replace')
        df = parse_kp_file(text)
        print(f'  Downloaded {len(df):,} rows  '
              f'({df.date.min().date()} → {df.date.max().date()})')
        return df
    except Exception as e:
        print(f'  Download failed: {e}')
        return None


def load_all_data(data_dir_path):
    
    data_dir = Path(data_dir_path)
    if not data_dir.exists():
        raise FileNotFoundError(
            f"Path '{data_dir}' does not exist. "
            "Set DATA_DIR = None to auto-download, or fix the path."
        )
    files = sorted(list(data_dir.glob('*.txt')) + list(data_dir.glob('*.TXT')))
    if not files:
        raise FileNotFoundError(f'No .txt files found in {data_dir}')

    print(f'Found {len(files)} file(s) in {data_dir}')
    dfs = []
    for fp in files:
        print(f'  Parsing: {fp.name}', end=' ')
        tmp = parse_kp_file(fp)
        if len(tmp) > 0:
            dfs.append(tmp)
            print(f'✓  ({len(tmp):,} rows)')
        else:
            print('✗ (no valid rows)')

    if not dfs:
        return pd.DataFrame()
    combined = pd.concat(dfs, ignore_index=True)
    combined = combined.drop_duplicates(subset='date').sort_values('date').reset_index(drop=True)
    print(f'\nTotal: {len(combined):,} rows  |  '
          f'{combined.date.min().date()} → {combined.date.max().date()}')
    return combined


df_raw = None

if DATA_DIR is not None:
    try:
        df_raw = load_all_data(DATA_DIR)
    except Exception as e:
        print(f'Local load failed: {e}')
        print('Falling back to auto-download...')

if df_raw is None or df_raw.empty:
    df_raw = download_gfz_data()

if df_raw is None or df_raw.empty:
    raise RuntimeError(
        'Could not load any data. '
        'Set DATA_DIR to a folder containing GFZ .txt files, '
        'or ensure internet access for auto-download.'
    )

print(df_raw.head(3))

## Output Directory Setup

In [ ]:

if DATA_DIR is not None and Path(DATA_DIR).exists():
    OUTPUT_DIR = Path(DATA_DIR) / 'v1-geo outputs'
else:
    OUTPUT_DIR = Path.cwd() / 'v1-geo outputs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR.resolve()}')

def save_fig(name):
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved → {path.name}')

## Feature Engineering

In [ ]:
def engineer_features(df):

    df = df.copy()
    kp_cols = [f'Kp{i}' for i in range(1, 9)]
    ap_cols = [f'ap{i}' for i in range(1, 9)]

    sentinel_cols = kp_cols + ap_cols + ['Ap', 'SN', 'F107obs', 'F107adj']
    for col in sentinel_cols:
        if col in df.columns:
            df[col] = df[col].replace({-1: np.nan, -1.0: np.nan})

    # Daily Kp stats 
    df['Kp_mean']  = df[kp_cols].mean(axis=1)
    df['Kp_max']   = df[kp_cols].max(axis=1)
    df['Kp_min']   = df[kp_cols].min(axis=1)
    df['Kp_std']   = df[kp_cols].std(axis=1).fillna(0)
    df['Kp_range'] = df['Kp_max'] - df['Kp_min']

    # Daily ap stats
    df['ap_mean']  = df[ap_cols].mean(axis=1)
    df['ap_max']   = df[ap_cols].max(axis=1)
    df['ap_std']   = df[ap_cols].std(axis=1).fillna(0)

    # Dst proxy
    df['Dst_proxy'] = -7.26 * df['Ap'] ** 0.52  # nT approx

    # Aurora flag
    df['aurora_flag']   = (df['Kp_max'] >= AURORA_KP_THRESHOLD).astype(float)
    # Storm level encoding
    df['storm_level'] = np.digitize(
        df['Kp_max'], bins=[0, 3, 4, 5, 6, 7, 9]
    ).astype(float)

    # Storm recovery feature
    storm_mask = df['Kp_max'] >= AURORA_KP_THRESHOLD
    storm_idx  = df.index[storm_mask].tolist()
    days_since = np.full(len(df), 999, dtype=float)
    last = -999
    for i in range(len(df)):
        if storm_mask.iloc[i]:
            last = i
        days_since[i] = i - last if last >= 0 else 999
    df['days_since_storm'] = np.clip(days_since, 0, 90)

    # Exponential moving averages (EMA) of Kp
    for span in [3, 7, 27]:
        df[f'Kp_ema{span}'] = df['Kp_mean'].ewm(span=span, adjust=False).mean()

    # Extended lag features (including 27-day solar rotation)
    for lag in [1, 2, 3, 7, 14, 27, 54]:
        df[f'Kp_mean_lag{lag}'] = df['Kp_mean'].shift(lag)
        df[f'Kp_max_lag{lag}']  = df['Kp_max'].shift(lag)
        df[f'Ap_lag{lag}']      = df['Ap'].shift(lag)

    for lag in [1, 3, 7, 27]:
        df[f'ap_mean_lag{lag}'] = df['ap_mean'].shift(lag)
        df[f'SN_lag{lag}']      = df['SN'].shift(lag)

    # Rolling statistics 
    for win in [3, 7, 14, 27]:
        df[f'Kp_roll{win}'] = df['Kp_mean'].rolling(win, min_periods=1).mean()
        df[f'Ap_roll{win}'] = df['Ap'].rolling(win, min_periods=1).mean()

    df['SN_roll27']   = df['SN'].rolling(27, min_periods=1).mean()
    df['F107_roll27'] = df['F107obs'].rolling(27, min_periods=1).mean()
    df['SN_roll81']   = df['SN'].rolling(81, min_periods=1).mean()
    df['F107_roll81'] = df['F107obs'].rolling(81, min_periods=1).mean()

    # Rolling standard deviations (volatility)
    df['Kp_vol7']  = df['Kp_mean'].rolling(7,  min_periods=2).std().fillna(0)
    df['Kp_vol27'] = df['Kp_mean'].rolling(27, min_periods=2).std().fillna(0)
    df['Ap_vol7']  = df['Ap'].rolling(7,  min_periods=2).std().fillna(0)

    # Interaction features
    df['SN_x_F107'] = df['SN_roll27'] * df['F107_roll27'] / 1e4
    df['Kp_x_Ap']   = df['Kp_mean'] * df['ap_mean'] / 100
    df['F107_SN_ratio'] = df['F107obs'] / (df['SN'] + 1) 

    # Solar cycle phase 
    df['year_frac']       = df['date'].dt.year + df['date'].dt.dayofyear / 365.25
    df['solar_cycle_sin'] = np.sin(2 * np.pi * (df['year_frac'] - 1996.5) / 11.3)
    df['solar_cycle_cos'] = np.cos(2 * np.pi * (df['year_frac'] - 1996.5) / 11.3)

    # Seasonal / calendar features 
    doy = df['date'].dt.dayofyear
    df['season_sin']  = np.sin(2 * np.pi * doy / 365.25)
    df['season_cos']  = np.cos(2 * np.pi * doy / 365.25)
    df['month_sin']   = np.sin(2 * np.pi * df['date'].dt.month / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['date'].dt.month / 12)

    # Equinox proximity (geomagnetic activity peaks near equinoxes
    df['equinox_prox'] = np.minimum(np.abs(doy - 80), np.abs(doy - 266)) / 90.0

    df = df.ffill().bfill().dropna().reset_index(drop=True)
    print(f'Feature-engineered shape: {df.shape}')
    return df

df = engineer_features(df_raw)

EXCLUDE = {'date', 'year_frac', 'F107adj'}
FEATURE_COLS = [
    c for c in df.columns
    if c not in EXCLUDE
    and c != TARGET_COL
    and pd.api.types.is_numeric_dtype(df[c])
]

print(f'\nUsing {len(FEATURE_COLS)} input features')
print(df[FEATURE_COLS[:10]].head(3))

## Exploratory Data Analysis

In [ ]:

fig, axes = plt.subplots(4, 1, figsize=(18, 14), sharex=True)
fig.suptitle('Geomagnetic & Solar Activity Overview', fontsize=16, fontweight='bold')

plot_df = df[df['date'] >= '1970-01-01'].copy()

ax = axes[0]
ax.plot(plot_df['date'], plot_df['Kp_mean'], lw=0.6, color='#3b82f6', label='Kp daily mean')
ax.axhline(5, color='#ef4444', lw=1, ls='--', label='G1 storm threshold')
ax.fill_between(plot_df['date'], 0, plot_df['Kp_mean'],
                where=plot_df['Kp_mean'] >= 5, alpha=0.3, color='#ef4444')
ax.set_ylabel('Kp'); ax.legend(fontsize=8); ax.set_ylim(0, 9.5)
ax.set_title('Daily Mean Kp Index', fontsize=11)

ax = axes[1]
ax.plot(plot_df['date'], plot_df['Ap'], lw=0.6, color='#10b981')
ax.set_ylabel('Ap (nT)'); ax.set_title('Daily Ap Index', fontsize=11)

ax = axes[2]
ax.plot(plot_df['date'], plot_df['SN'], lw=0.4, color='#f59e0b', alpha=0.5, label='Daily SN')
ax.plot(plot_df['date'], plot_df['SN_roll27'], lw=1.2, color='#d97706', label='27-day mean')
ax.set_ylabel('SN'); ax.legend(fontsize=8); ax.set_title('Sunspot Number', fontsize=11)

ax = axes[3]
mask = plot_df['F107obs'] > 0
ax.plot(plot_df.loc[mask, 'date'], plot_df.loc[mask, 'F107obs'], lw=0.6, color='#8b5cf6')
ax.set_ylabel('F10.7 (s.f.u.)'); ax.set_xlabel('Date')
ax.set_title('F10.7 Solar Radio Flux', fontsize=11)

ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=30)
plt.tight_layout()
save_fig('aurora_lr_eda_overview.png')
plt.show()

aurora_days = df['aurora_flag'].sum()
print(f'Total days   : {len(df):,}')
print(f'Aurora days  : {int(aurora_days):,}  ({100*aurora_days/len(df):.1f}%)')
print(f'Kp mean±std  : {df["Kp_mean"].mean():.2f} ± {df["Kp_mean"].std():.2f}')

In [ ]:

corr_cols = ['Kp_mean'] + [
    'Kp_mean_lag1', 'Kp_mean_lag3', 'Kp_mean_lag7', 'Kp_mean_lag27',
    'Kp_max_lag1', 'Kp_roll7', 'Ap_roll7', 'Kp_ema7',
    'SN_roll27', 'F107_roll27', 'solar_cycle_sin', 'equinox_prox',
    'days_since_storm'
]
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('aurora_lr_correlation.png')
plt.show()

## Train / Validation / Test Split

In [ ]:
def build_lagged_dataset(df, feature_cols, target_col, lookback):

    X_base = df[feature_cols].values
    y_base = df[target_col].values

    lag_names  = []
    lag_arrays = []
    for lag in range(1, lookback + 1):
        lagged = pd.Series(y_base).shift(lag).values
        lag_arrays.append(lagged)
        lag_names.append(f'target_lag{lag}')

    lag_matrix = np.column_stack(lag_arrays)
    X_full     = np.hstack([X_base, lag_matrix])

    valid  = ~np.isnan(X_full).any(axis=1)
    X_full = X_full[valid]
    y_out  = y_base[valid]
    dates  = df['date'].values[valid]

    col_names = feature_cols + lag_names
    return X_full, y_out, dates, col_names


X_all, y_all, dates_all, col_names = build_lagged_dataset(
    df, FEATURE_COLS, TARGET_COL, LOOKBACK
)

n        = len(X_all)
n_test   = int(n * TEST_SPLIT)
n_val    = int(n * VAL_SPLIT)
n_train  = n - n_test - n_val

X_train, y_train = X_all[:n_train],              y_all[:n_train]
X_val,   y_val   = X_all[n_train:n_train+n_val], y_all[n_train:n_train+n_val]
X_test,  y_test  = X_all[n_train+n_val:],        y_all[n_train+n_val:]
dates_test       = dates_all[n_train+n_val:]

scaler    = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f'Train : {X_train_s.shape}  ({dates_all[0].astype("datetime64[D]")} → )')
print(f'Val   : {X_val_s.shape}')
print(f'Test  : {X_test_s.shape}')
print(f'Features per sample: {X_train_s.shape[1]}')

## Hyperparameter Tuning & Model Training

In [ ]:

tscv_tune = TimeSeriesSplit(n_splits=4)
X_tv = np.vstack([X_train_s, X_val_s])
y_tv = np.concatenate([y_train, y_val])

print('Tuning hyperparameters (train+val, 4-fold TSCV)...')

ridge_gs = GridSearchCV(
    Ridge(), {'alpha': [0.01, 0.1, 1, 5, 10, 50, 100]},
    cv=tscv_tune, scoring='r2', n_jobs=-1
).fit(X_tv, y_tv)
best_ridge_alpha = ridge_gs.best_params_['alpha']
print(f'  Best Ridge alpha : {best_ridge_alpha}  (CV R²={ridge_gs.best_score_:.4f})')

lasso_gs = GridSearchCV(
    Lasso(max_iter=5000), {'alpha': [0.001, 0.005, 0.01, 0.05, 0.1]},
    cv=tscv_tune, scoring='r2', n_jobs=-1
).fit(X_tv, y_tv)
best_lasso_alpha = lasso_gs.best_params_['alpha']
print(f'  Best Lasso alpha : {best_lasso_alpha}  (CV R²={lasso_gs.best_score_:.4f})')

en_gs = GridSearchCV(
    ElasticNet(max_iter=5000),
    {'alpha': [0.001, 0.01, 0.05], 'l1_ratio': [0.2, 0.5, 0.8]},
    cv=tscv_tune, scoring='r2', n_jobs=-1
).fit(X_tv, y_tv)
best_en_alpha    = en_gs.best_params_['alpha']
best_en_l1ratio  = en_gs.best_params_['l1_ratio']
print(f'  Best ElasticNet  : alpha={best_en_alpha} l1_ratio={best_en_l1ratio}  '
      f'(CV R²={en_gs.best_score_:.4f})')

print('Hyperparameter tuning complete ✓')

In [ ]:

model_definitions = {
    'Linear Regression': LinearRegression(fit_intercept=True),
    'Ridge':             Ridge(alpha=best_ridge_alpha, fit_intercept=True),
    'Lasso':             Lasso(alpha=best_lasso_alpha, fit_intercept=True, max_iter=5000),
    'ElasticNet':        ElasticNet(alpha=best_en_alpha, l1_ratio=best_en_l1ratio,
                                    fit_intercept=True, max_iter=5000),
    'Polynomial Ridge':  Pipeline([
        ('poly',  PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
        ('ridge', Ridge(alpha=best_ridge_alpha * 10))
    ]),
    'Huber':             HuberRegressor(epsilon=1.35, alpha=0.001, max_iter=300),
    'BayesianRidge':     BayesianRidge(max_iter=300),
}

trained_models = {}
print('Training models...')
print('=' * 65)
for name, model in model_definitions.items():
    print(f'  Fitting: {name}', end='  ')
    model.fit(X_train_s, y_train)
    train_r2 = r2_score(y_train, model.predict(X_train_s))
    val_r2   = r2_score(y_val,   model.predict(X_val_s))
    print(f'Train R²={train_r2:.4f}  Val R²={val_r2:.4f}')
    trained_models[name] = model

print('\nAll models trained.')

## Evaluation on Test Set

In [ ]:
def evaluate_model(model, X_te, y_te, name):
    y_pred = model.predict(X_te)
    y_pred = np.clip(y_pred, 0, 9)

    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    mape = np.mean(np.abs((y_te - y_pred) / (y_te + 1e-8))) * 100

    true_aurora = (y_te   >= AURORA_KP_THRESHOLD).astype(int)
    pred_aurora = (y_pred >= AURORA_KP_THRESHOLD).astype(int)
    prec = precision_score(true_aurora, pred_aurora, zero_division=0)
    rec  = recall_score(true_aurora, pred_aurora, zero_division=0)
    f1   = f1_score(true_aurora, pred_aurora, zero_division=0)

    print(f'  {name:<22}  RMSE:{rmse:.4f}  MAE:{mae:.4f}  R²:{r2:.4f}  '
          f'MAPE:{mape:.1f}%  |  Aurora  Prec:{prec:.3f}  Rec:{rec:.3f}  F1:{f1:.3f}')
    return {
        'name': name, 'y_pred': y_pred, 'y_true': y_te,
        'rmse': rmse, 'mae': mae, 'r2': r2, 'mape': mape,
        'precision': prec, 'recall': rec, 'f1': f1
    }


print('TEST SET METRICS')
print('=' * 120)
eval_results = {}
for name, model in trained_models.items():
    eval_results[name] = evaluate_model(model, X_test_s, y_test, name)

## Calibrated Aurora Probability

In [ ]:
# Calibrate aurora probability using isotonic regression
# Best model by R²
best_name  = max(eval_results, key=lambda k: eval_results[k]['r2'])
best_model = trained_models[best_name]
print(f'Best model: {best_name}  (R²={eval_results[best_name]["r2"]:.4f})')

# Raw score = predicted Kp; calibrate P(aurora) with isotonic regression
val_pred_raw = np.clip(best_model.predict(X_val_s), 0, 9)
val_aurora   = (y_val >= AURORA_KP_THRESHOLD).astype(float)
calibrator   = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(val_pred_raw, val_aurora)

test_pred_raw = np.clip(best_model.predict(X_test_s), 0, 9)
test_aurora_prob = calibrator.predict(test_pred_raw)
test_aurora_true = (y_test >= AURORA_KP_THRESHOLD).astype(float)

brier = brier_score_loss(test_aurora_true, test_aurora_prob)
print(f'Calibrated Brier score (aurora Kp≥5): {brier:.4f}  (lower = better)')
print('Calibrator trained ✓')

## Predictions vs Actuals

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(22, 24))
fig.suptitle('Kp Index — Predictions vs Actuals (Test Set)', fontsize=15, fontweight='bold')

for ax, (name, res) in zip(axes.ravel(), eval_results.items()):
    y_true = res['y_true']
    y_pred = res['y_pred']
    color  = COLORS.get(name, '#6366f1')

    ax.plot(dates_test, y_true, lw=1.0, color=COLORS['Actual'], label='Actual Kp', alpha=0.9, zorder=4)
    ax.plot(dates_test, y_pred, lw=0.9, color=color, label=name, alpha=0.85, ls='--', zorder=3)
    ax.axhline(AURORA_KP_THRESHOLD, color='#dc2626', lw=0.8, ls=':', alpha=0.5)
    ax.fill_between(dates_test, AURORA_KP_THRESHOLD, 9.5,
                    where=(y_true >= AURORA_KP_THRESHOLD),
                    alpha=0.12, color='#ef4444', label='Aurora events')
    ax.set_title(
        f"{name}  |  R²={res['r2']:.4f}  RMSE={res['rmse']:.4f}  Aurora F1={res['f1']:.3f}",
        fontsize=9, fontweight='bold'
    )
    ax.set_ylabel('Kp Index')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(-0.3, 9.5)
    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Hide any unused axes
for ax in axes.ravel()[len(eval_results):]:
    ax.set_visible(False)

plt.tight_layout()
save_fig('aurora_lr_predictions_vs_actuals.png')
plt.show()

In [ ]:
# Residual distributions
fig, axes = plt.subplots(4, 2, figsize=(18, 16))
fig.suptitle('Residual Distributions by Model', fontsize=14, fontweight='bold')

for ax, (name, res) in zip(axes.ravel(), eval_results.items()):
    residuals = res['y_true'] - res['y_pred']
    color = COLORS.get(name, '#6366f1')
    ax.hist(residuals, bins=60, color=color, alpha=0.75, edgecolor='white', lw=0.4)
    ax.axvline(0, color='black', lw=1.2, ls='--')
    ax.axvline(residuals.mean(), color='red', lw=1, ls='--',
               label=f'Mean={residuals.mean():.3f}')
    ax.set_title(f"{name}  (σ={residuals.std():.3f})", fontsize=10, fontweight='bold')
    ax.set_xlabel('Residual (Actual − Predicted Kp)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

for ax in axes.ravel()[len(eval_results):]:
    ax.set_visible(False)

plt.tight_layout()
save_fig('aurora_lr_residuals.png')
plt.show()

In [ ]:
# Scatter: predicted vs actual
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
fig.suptitle('Predicted vs Actual Kp — Scatter Plots', fontsize=14, fontweight='bold')

for ax, (name, res) in zip(axes.ravel(), eval_results.items()):
    color = COLORS.get(name, '#6366f1')
    ax.scatter(res['y_true'], res['y_pred'], alpha=0.15, s=4, color=color)
    lims = [0, 9]
    ax.plot(lims, lims, 'k--', lw=1, label='Perfect prediction')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Actual Kp'); ax.set_ylabel('Predicted Kp')
    ax.set_title(f"{name}  R²={res['r2']:.4f}", fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

for ax in axes.ravel()[len(eval_results):]:
    ax.set_visible(False)

plt.tight_layout()
save_fig('aurora_lr_scatter.png')
plt.show()

## Model Comparison Dashboard

In [ ]:
rows = []
for name, res in eval_results.items():
    rows.append({
        'Model':       name,
        'RMSE':        round(res['rmse'], 4),
        'MAE':         round(res['mae'], 4),
        'R²':          round(res['r2'], 4),
        'MAPE (%)':    round(res['mape'], 2),
        'Aurora Prec': round(res['precision'], 3),
        'Aurora Rec':  round(res['recall'], 3),
        'Aurora F1':   round(res['f1'], 3),
    })
comp_df = pd.DataFrame(rows).sort_values('R²', ascending=False)
print('\nMODEL COMPARISON TABLE')
print(comp_df.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Linear Model Comparison — Aurora Prediction Metrics', fontsize=14, fontweight='bold')

metrics_plot = [
    ('R²',        True,  'R² (higher = better)'),
    ('RMSE',      False, 'RMSE (lower = better)'),
    ('MAE',       False, 'MAE (lower = better)'),
    ('MAPE (%)',  False, 'MAPE % (lower = better)'),
    ('Aurora F1', True,  'Aurora F1 (higher = better)'),
    ('Aurora Rec',True,  'Aurora Recall (higher = better)'),
]

for ax, (metric, higher, title) in zip(axes.ravel(), metrics_plot):
    sdf = comp_df.sort_values(metric, ascending=not higher)
    bar_colors = [COLORS.get(n, '#6366f1') for n in sdf['Model']]
    ax.barh(sdf['Model'], sdf[metric], color=bar_colors, alpha=0.85, edgecolor='white')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel(metric)
    for i, (_, row) in enumerate(sdf.iterrows()):
        ax.text(row[metric] * 0.02, i, f"{row[metric]:.4f}", va='center', fontsize=8)

plt.tight_layout()
save_fig('aurora_lr_model_comparison.png')
plt.show()

## Time-Series Cross-Validation

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_results = {}
print('Time-Series Cross-Validation (5 folds)')
print('=' * 65)

X_cv = np.vstack([X_train_s, X_val_s])
y_cv = np.concatenate([y_train, y_val])

for name, model in trained_models.items():
    scores = cross_val_score(model, X_cv, y_cv, cv=tscv, scoring='r2', n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name:<22}  CV R²: {scores.mean():.4f} ± {scores.std():.4f}  '
          f'folds: {np.round(scores, 3)}')

fig, ax = plt.subplots(figsize=(12, 6))
cv_data   = [cv_results[n] for n in trained_models]
cv_labels = list(trained_models.keys())
bp = ax.boxplot(cv_data, labels=cv_labels, patch_artist=True, notch=False)
for patch, name in zip(bp['boxes'], cv_labels):
    patch.set_facecolor(COLORS.get(name, '#6366f1'))
    patch.set_alpha(0.75)
ax.axhline(0, color='gray', ls='--', lw=1)
ax.set_ylabel('R² Score'); ax.set_xlabel('Model')
ax.set_title('Time-Series Cross-Validation R² (5 folds)', fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
save_fig('aurora_lr_crossval.png')
plt.show()

## Feature Importance

In [ ]:
if isinstance(best_model, Pipeline):
    coefs   = best_model.named_steps['ridge'].coef_
    use_perm = True
else:
    coefs   = best_model.coef_ if hasattr(best_model, 'coef_') else None
    use_perm = (coefs is None)

if coefs is not None and not use_perm:
    top_n     = 25
    imp       = np.abs(coefs)
    top_idx   = np.argsort(imp)[-top_n:][::-1]
    top_names = [col_names[i] if i < len(col_names) else f'feat_{i}' for i in top_idx]
    top_vals  = imp[top_idx]
    top_signs = np.sign(coefs[top_idx])
    bar_colors = ['#10b981' if s > 0 else '#ef4444' for s in top_signs]

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(top_names[::-1], top_vals[::-1] * top_signs[::-1],
            color=bar_colors[::-1], alpha=0.85)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Coefficient (signed magnitude)')
    ax.set_title(f'Top-{top_n} Feature Importances  [{best_name}]\n'
                 'Green = positive effect on Kp, Red = negative',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    save_fig('aurora_lr_feature_importance.png')
    plt.show()

print('\nComputing permutation importances (may take ~1–2 min)...')
perm      = permutation_importance(
    best_model, X_test_s, y_test,
    n_repeats=15, random_state=42, scoring='r2'
)
perm_imp  = perm.importances_mean
perm_std  = perm.importances_std
top_pi    = np.argsort(perm_imp)[-25:][::-1]
pi_names  = [col_names[i] if i < len(col_names) else f'feat_{i}' for i in top_pi]
pi_vals   = perm_imp[top_pi]
pi_errs   = perm_std[top_pi]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(pi_names[::-1], pi_vals[::-1], xerr=pi_errs[::-1],
        color='#3b82f6', alpha=0.85, capsize=3)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Mean decrease in R² when feature is permuted')
ax.set_title(f'Permutation Feature Importance  [{best_name}]',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_fig('aurora_lr_perm_importance.png')
plt.show()

## 30-Day Aurora Forecast

In [ ]:
def recursive_forecast_lr(model, df_tail, feature_cols, col_names, scaler,
                           lookback, steps):
    history = list(df_tail[TARGET_COL].values[-(lookback + 30):])
    feature_history = df_tail[feature_cols].values.copy()
    forecasts = []

    for step in range(steps):
        base_features = feature_history[-1].copy()

        # Build lag vector for target column
        target_lags = np.array([history[-(lag)] for lag in range(1, lookback + 1)])

        x_new   = np.concatenate([base_features, target_lags]).reshape(1, -1)
        x_new_s = scaler.transform(x_new)

        y_pred = float(model.predict(x_new_s)[0])
        y_pred = np.clip(y_pred, 0, 9)
        forecasts.append(y_pred)
        history.append(y_pred)

        # Build updated feature row: shift Kp-based rolling features
        next_row = feature_history[-1].copy()
        # Update EMA proxies where columns exist
        for fname in feature_cols:
            if 'ema' in fname or 'roll' in fname or 'lag1' in fname:
                idx = feature_cols.index(fname)
                if 'lag1' in fname and 'Kp' in fname:
                    next_row[idx] = y_pred
        feature_history = np.vstack([feature_history, next_row])

    return np.array(forecasts)


last_date      = df['date'].iloc[-1]
forecast_dates = pd.date_range(last_date + timedelta(days=1),
                               periods=FORECAST_DAYS, freq='D')

forecasts = {}
print('Generating 30-day forecasts...')
for name, model in trained_models.items():
    fc = recursive_forecast_lr(model, df, FEATURE_COLS, col_names, scaler,
                                LOOKBACK, FORECAST_DAYS)
    forecasts[name] = fc
    print(f'  {name:<22}: mean={fc.mean():.2f}  max={fc.max():.2f}  '
          f'aurora_days={int((fc >= AURORA_KP_THRESHOLD).sum())}')

cv_means = {n: cv_results[n].mean() for n in trained_models}
min_cv   = min(cv_means.values())
# Shift so all weights ≥ 0
shifted  = {n: max(v - min_cv, 0) + 1e-6 for n, v in cv_means.items()}
total_w  = sum(shifted.values())
weights  = {n: v / total_w for n, v in shifted.items()}

ensemble_fc = np.zeros(FORECAST_DAYS)
for name in trained_models:
    ensemble_fc += weights[name] * forecasts[name]
forecasts['Ensemble (weighted)'] = ensemble_fc

print(f'\nEnsemble (weighted): mean={ensemble_fc.mean():.2f}  '
      f'aurora_days={int((ensemble_fc >= AURORA_KP_THRESHOLD).sum())}')

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(16, 12))
fig.suptitle(
    f'30-Day Aurora / Kp Forecast  ({forecast_dates[0].date()} → {forecast_dates[-1].date()})',
    fontsize=14, fontweight='bold'
)

context_df = df.iloc[-90:]
ctx_dates  = context_df['date'].values
ctx_kp     = context_df[TARGET_COL].values

for ax in axes:
    ax.plot(ctx_dates, ctx_kp, color=COLORS['Actual'], lw=1.5, label='Historical Kp', zorder=5)
    ax.axvline(last_date, color='black', ls='--', lw=1, alpha=0.5, label='Forecast start')
    ax.axhspan(AURORA_KP_THRESHOLD, 9.5, alpha=0.07, color='#ef4444',
               label=f'Aurora zone (Kp≥{AURORA_KP_THRESHOLD})')

for name, fc in forecasts.items():
    if name == 'Ensemble (weighted)':
        continue
    axes[0].plot(forecast_dates, fc, color=COLORS.get(name, '#6366f1'),
                 lw=1.2, ls='--', label=name, alpha=0.85)
axes[0].set_title('Individual Model Forecasts', fontsize=11)
axes[0].legend(fontsize=9); axes[0].set_ylabel('Kp Index'); axes[0].set_ylim(-0.3, 9.5)

all_fc_arr = np.array([forecasts[n] for n in trained_models])
fc_lower   = np.percentile(all_fc_arr, 10, axis=0)
fc_upper   = np.percentile(all_fc_arr, 90, axis=0)
axes[1].fill_between(forecast_dates, fc_lower, fc_upper,
                     alpha=0.25, color='#3b82f6', label='10–90th pct band')
axes[1].plot(forecast_dates, ensemble_fc, color=COLORS['Actual'],
             lw=2, label='Weighted Ensemble forecast')
axes[1].set_title('Weighted Ensemble Forecast with Uncertainty Band', fontsize=11)
axes[1].legend(fontsize=9); axes[1].set_ylabel('Kp Index'); axes[1].set_ylim(-0.3, 9.5)
axes[1].set_xlabel('Date')

for ax in axes:
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
save_fig('aurora_lr_30day_forecast.png')
plt.show()

## NOAA / NASA Forecast Comparison

This section fetches the official NOAA Space Weather Prediction Center (SWPC) 3-day Kp forecast and compares it against the model ensemble. If internet access is unavailable, a reference table of typical NOAA forecast values is shown instead.

In [ ]:
def fetch_noaa_kp_forecast():

    url = 'https://services.swpc.noaa.gov/products/noaa-planetary-k-index-forecast.json'
    try:
        if REQUESTS_OK:
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            data = resp.json()
        else:
            with urllib.request.urlopen(url, timeout=15) as r:
                import json
                data = json.loads(r.read().decode())

        # data[0] is the header row
        rows = []
        for row in data[1:]:
            try:
                dt = pd.to_datetime(row[0])
                kp = float(row[1])
                rows.append({'datetime': dt, 'kp_noaa': kp})
            except Exception:
                continue

        if not rows:
            return None

        df_noaa = pd.DataFrame(rows)
        # Aggregate to daily mean
        df_noaa['date'] = df_noaa['datetime'].dt.normalize()
        daily_noaa = (
            df_noaa.groupby('date')['kp_noaa']
            .mean()
            .reset_index()
            .rename(columns={'kp_noaa': 'Kp_NOAA'})
        )
        print(f'NOAA forecast: {len(daily_noaa)} days fetched ({daily_noaa.date.min().date()} → {daily_noaa.date.max().date()})')
        return daily_noaa

    except Exception as e:
        print(f'NOAA fetch failed: {e}')
        return None


print('Fetching NOAA SWPC Kp forecast...')
noaa_fc = fetch_noaa_kp_forecast()

if noaa_fc is not None:
    # Align with our forecast dates
    fc_series = pd.DataFrame({
        'date': forecast_dates,
        'Kp_Ensemble': ensemble_fc
    })
    comparison = fc_series.merge(noaa_fc, on='date', how='inner')

    if len(comparison) > 0:
        rmse_vs_noaa = np.sqrt(mean_squared_error(comparison['Kp_NOAA'], comparison['Kp_Ensemble']))
        mae_vs_noaa  = mean_absolute_error(comparison['Kp_NOAA'], comparison['Kp_Ensemble'])

        fig, axes = plt.subplots(2, 1, figsize=(14, 10))
        fig.suptitle('Model Ensemble vs NOAA SWPC Official Forecast',
                     fontsize=14, fontweight='bold')

        ax = axes[0]
        ax.plot(comparison['date'], comparison['Kp_NOAA'],
                color='#ef4444', lw=2, marker='o', ms=5, label='NOAA SWPC Official')
        ax.plot(comparison['date'], comparison['Kp_Ensemble'],
                color='#3b82f6', lw=2, ls='--', marker='s', ms=4, label='Our Weighted Ensemble')
        ax.axhline(AURORA_KP_THRESHOLD, color='gray', lw=1, ls=':', alpha=0.6)
        ax.set_ylabel('Kp Index'); ax.set_title('Kp Forecast Comparison')
        ax.legend(fontsize=10); ax.set_ylim(-0.3, 9.5)
        ax.text(0.02, 0.95,
                f'RMSE vs NOAA: {rmse_vs_noaa:.3f}\nMAE vs NOAA: {mae_vs_noaa:.3f}',
                transform=ax.transAxes, va='top', fontsize=10,
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))

        ax = axes[1]
        diff = comparison['Kp_Ensemble'] - comparison['Kp_NOAA']
        ax.bar(comparison['date'], diff,
               color=np.where(diff >= 0, '#10b981', '#ef4444'), alpha=0.8)
        ax.axhline(0, color='black', lw=1)
        ax.set_ylabel('Ensemble − NOAA  (Kp)'); ax.set_xlabel('Date')
        ax.set_title('Difference: Our Forecast minus NOAA')

        for ax in axes:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

        plt.tight_layout()
        save_fig('aurora_lr_noaa_comparison.png')
        plt.show()

        print(f'\n📡 Overlap with NOAA: {len(comparison)} days')
        print(f'   RMSE vs NOAA official: {rmse_vs_noaa:.4f}')
        print(f'   MAE  vs NOAA official: {mae_vs_noaa:.4f}')
        print(comparison[['date', 'Kp_NOAA', 'Kp_Ensemble']].to_string(index=False))
    else:
        print('No overlapping dates between our forecast period and NOAA data.')
        print('(NOAA only covers ±3 days from today; our forecast starts after training data ends.)')

else:
    print()
    print('─' * 72)
    print('📡 NOAA fetch unavailable — showing reference NOAA forecast scale:')
    print('─' * 72)
    print('NOAA issues daily 3-day Kp forecasts at:')
    print('  https://www.swpc.noaa.gov/products/3-day-forecast')
    print()
    print('Kp Storm Levels (NOAA scale):')
    print('  Kp 5–6  → G1  Minor geomagnetic storm')
    print('  Kp 6–7  → G2  Moderate geomagnetic storm')
    print('  Kp 7–8  → G3  Strong geomagnetic storm')
    print('  Kp 8–9  → G4  Severe geomagnetic storm')
    print('  Kp 9    → G5  Extreme geomagnetic storm')
    print()
    print('Our ensemble 30-day forecast summary:')
    print(f'  Mean Kp : {ensemble_fc.mean():.2f}')
    print(f'  Max Kp  : {ensemble_fc.max():.2f}')
    print(f'  Aurora days (Kp≥5): {int((ensemble_fc >= AURORA_KP_THRESHOLD).sum())}')
    print()
    print('To enable live NOAA comparison, ensure internet access when running this notebook.')

## Aurora Alert Calendar

In [ ]:
all_fc_arr   = np.array([forecasts[n] for n in trained_models])
aurora_prob  = (all_fc_arr >= AURORA_KP_THRESHOLD).mean(axis=0)

# Calibrate ensemble probability using isotonic calibrator
aurora_prob_cal = calibrator.predict(ensemble_fc)

alert_df = pd.DataFrame({
    'Date':              forecast_dates,
    'Kp_Ensemble':       np.round(ensemble_fc, 2),
    'Kp_Lower_10':       np.round(fc_lower, 2),
    'Kp_Upper_90':       np.round(fc_upper, 2),
    'Aurora_Prob_Raw%':  np.round(aurora_prob * 100, 1),
    'Aurora_Prob_Cal%':  np.round(aurora_prob_cal * 100, 1),
    'Storm_Level':       ['G3+' if k >= 7 else 'G2' if k >= 6
                          else 'G1' if k >= 5 else 'Active' if k >= 4
                          else 'Quiet' for k in ensemble_fc]
})

print('\n30-DAY AURORA FORECAST CALENDAR')
print('=' * 80)
print(alert_df.to_string(index=False))

# Calendar heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('30-Day Aurora Alert Calendar (Weighted Ensemble)', fontsize=13, fontweight='bold')

im1 = axes[0].imshow(ensemble_fc.reshape(1, -1), cmap='RdYlGn_r', vmin=0, vmax=9, aspect='auto')
axes[0].set_xticks(range(FORECAST_DAYS))
axes[0].set_xticklabels([d.strftime('%d %b') for d in forecast_dates], rotation=60, fontsize=7)
axes[0].set_yticks([])
axes[0].set_title('Ensemble Kp Forecast')
for j, kp in enumerate(ensemble_fc):
    axes[0].text(j, 0, f'{kp:.1f}', ha='center', va='center', fontsize=7,
                 color='white' if kp > 4 else 'black', fontweight='bold')
plt.colorbar(im1, ax=axes[0], label='Kp')

im2 = axes[1].imshow(aurora_prob_cal.reshape(1, -1), cmap='Reds', vmin=0, vmax=1, aspect='auto')
axes[1].set_xticks(range(FORECAST_DAYS))
axes[1].set_xticklabels([d.strftime('%d %b') for d in forecast_dates], rotation=60, fontsize=7)
axes[1].set_yticks([])
axes[1].set_title('Aurora Probability — Calibrated (Kp ≥ 5)')
for j, p in enumerate(aurora_prob_cal):
    axes[1].text(j, 0, f'{p*100:.0f}%', ha='center', va='center', fontsize=7,
                 color='white' if p > 0.5 else 'black', fontweight='bold')
plt.colorbar(im2, ax=axes[1], label='Probability')

plt.tight_layout()
save_fig('aurora_lr_alert_calendar.png')
plt.show()

## Regularisation Path (Ridge / Lasso)

In [ ]:
alphas = np.logspace(-3, 3, 40)

ridge_scores, lasso_scores = [], []
for a in alphas:
    r = Ridge(alpha=a).fit(X_train_s, y_train)
    ridge_scores.append(r2_score(y_test, np.clip(r.predict(X_test_s), 0, 9)))

    l = Lasso(alpha=a, max_iter=3000).fit(X_train_s, y_train)
    lasso_scores.append(r2_score(y_test, np.clip(l.predict(X_test_s), 0, 9)))

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogx(alphas, ridge_scores, color='#10b981', lw=2, label='Ridge R²')
ax.semilogx(alphas, lasso_scores, color='#f59e0b', lw=2, label='Lasso R²')
ax.axvline(best_ridge_alpha, color='#10b981', ls='--', lw=1,
           label=f'Tuned Ridge α={best_ridge_alpha}')
ax.axvline(best_lasso_alpha, color='#f59e0b', ls='--', lw=1,
           label=f'Tuned Lasso α={best_lasso_alpha:.4f}')
ax.set_xlabel('Alpha (regularisation strength)'); ax.set_ylabel('Test R²')
ax.set_title('Regularisation Path: Test R² vs Alpha', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig('aurora_lr_reg_path.png')
plt.show()

print(f'Best Ridge alpha : {alphas[np.argmax(ridge_scores)]:.4f}  '
      f'(R²={max(ridge_scores):.4f})')
print(f'Best Lasso alpha : {alphas[np.argmax(lasso_scores)]:.4f}  '
      f'(R²={max(lasso_scores):.4f})')

## Save All Outputs

In [ ]:

joblib.dump(scaler,     OUTPUT_DIR / 'aurora_lr_feature_scaler.pkl')
joblib.dump(calibrator, OUTPUT_DIR / 'aurora_lr_aurora_calibrator.pkl')
print('Scaler & calibrator saved.')

for name, model in trained_models.items():
    fname = f"aurora_lr_{name.replace(' ', '_')}.pkl"
    joblib.dump(model, OUTPUT_DIR / fname)
    print(f'Model saved: {fname}')

alert_df.to_csv(OUTPUT_DIR / 'aurora_lr_30day_forecast.csv', index=False)
print('Forecast CSV saved.')

comp_df.to_csv(OUTPUT_DIR / 'aurora_lr_model_metrics.csv', index=False)
print('Metrics CSV saved.')

# Save ensemble weights
weights_df = pd.DataFrame([
    {'Model': k, 'Weight': round(v, 6), 'CV_R2': round(cv_means[k], 4)}
    for k, v in weights.items()
])
weights_df.to_csv(OUTPUT_DIR / 'aurora_lr_ensemble_weights.csv', index=False)
print('Ensemble weights CSV saved.')

print(f'\nAll outputs saved to: {OUTPUT_DIR.resolve()}')
print('\nOutput files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name}')